<a href="https://colab.research.google.com/github/anytaaly/Cershe-GRU-training/blob/main/Cershe_GRU_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()

Saving claude-drafts-1000.txt to claude-drafts-1000.txt


In [2]:
uploaded = files.upload()


Saving stories-draft-001.txt to stories-draft-001.txt


#read every story into training format

In [10]:
import re
blocks = []

#read each file separately and cuts it into stories at every SEQUENCE:. Reading them separately stops the format notes at the top of your file from leaking into my last story.
for f in ['claude-drafts-1000.txt', 'stories-draft-001.txt']:
  blocks += open(f, encoding='utf-8').read().split('SEQUENCE:')[1:]

#define a lambda function
num = lambda x:float(x.replace('-','-'))


def field(part, name, default=''):
  m = re.search(rf'^{name}: *(.*)$', part, re.M)
  return m.group(1).strip() if m else default

stories = []
for b in blocks:
    head, *steps = re.split(r'^STEP \d+\s*$', b, flags=re.M)
    story = {'name': b.split()[0],
             'start': (num(field(head, 'Starting valence', '0.10')), num(field(head, 'Starting arousal', '0.20'))),
             'steps': []}
    for s in steps:
        story['steps'].append({
            'minutes': int(field(s, 'Elapsed minutes since previous step', '0')),
            'type': field(s, 'Event type'),
            'event': field(s, 'Event'),
            'core': int(field(s, 'Core value related', '0')),
            'opens': 1 if field(s, 'Unresolved') else 0,
            'resolves': 1 if field(s, 'Resolves') else 0,
            'valence': num(field(s, 'Target valence')),
            'arousal': num(field(s, 'Target arousal')),
        })
    stories.append(story)

print(len(stories), 'stories,', sum(len(st['steps']) for st in stories), 'steps')
print('opens:', sum(s['opens'] for st in stories for s in st['steps']), 'resolves:', sum(s['resolves'] for st in stories for s in st['steps']))


1009 stories, 4847 steps
opens: 488 resolves: 534


#turn each event sentence into numbers with MiniLM

In [11]:
!pip -q install sentence-transformers
from sentence_transformers import SentenceTransformer

In [12]:
#load a pre-trained sentence embedding model and stores it in the variable encoder.
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
sentences = sorted({s['event'] for st in stories for s in st['steps']})

#encoder.encode(...) turns each sentence into 384 numbers that capture its meaning. normalize_embeddings=True puts them all on the same scale so they can be compared.
vectors = encoder.encode(sentences, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
vec = dict(zip(sentences, vectors))


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [14]:
print(len(sentences), 'different sentences ->', vectors.shape)

2558 different sentences -> (2558, 384)


In [15]:
pairs = [('The school responded to the bullying.', 'The school has not responded to the bullying.'),
         ('The vet says the lump is benign.', 'The vet says the lump is malignant.'),
         ('The dog was found safe.', 'The dog was found dead.'),
         ('The dog was found safe.', 'It rained all afternoon.')]
for a, b in pairs: print(round(float(vec.get(a, encoder.encode(a, normalize_embeddings=True)) @ encoder.encode(b, normalize_embeddings=True)), 2), '|', a, '|', b)

0.68 | The school responded to the bullying. | The school has not responded to the bullying.
0.88 | The vet says the lump is benign. | The vet says the lump is malignant.
0.68 | The dog was found safe. | The dog was found dead.
0.16 | The dog was found safe. | It rained all afternoon.
